# Introducción al aprendizaje automatizado (O2026)
## Tutorial 13 - Bonus: Conditional Inference Trees en Python (`citrees`)

**Objetivo:**  
Mostrar **por qué existe** `ctree` / `citrees` como alternativa a CART. Vemos primero el problema teórico que vienen a resolver (el *sesgo de selección de variables* de CART) con una simulación didáctica, después comparamos los dos métodos en datos reales (Boston Housing), y cerramos con una aplicación reciente en economía.

### Temario:
1. ¿Qué es un *Conditional Inference Tree* y en qué se diferencia de CART?
2. ¿Existe algo equivalente en Python? El paquete `citrees`.
3. **Actividad 1:** demostración del sesgo de selección de CART con datos simulados.
4. **Actividad 2:** CART vs CIT sobre Boston Housing.
5. Aplicación en economía: Huo & Feng (2024, IZA DP 16764) sobre desigualdad de oportunidades en salud.


> **Nota sobre cómo usar este notebook.**
>
> Este notebook está pensado para **leerse con los outputs ya ejecutados**: los resultados de cada celda están guardados, no hace falta correr el código para entender el contenido.
>
> Para **correrlo localmente** se necesita un setup específico: Python ≥ 3.12, R instalado en el sistema, y un *venv* aislado con el paquete `citrees` clonado desde GitHub. Las instrucciones completas están en la **sección 2 (Instalación)**.
>
> Para los TPs y la cursada **no se requiere reproducir este notebook**; es material de referencia para conocer la herramienta y su aplicación en economía.


## 1. ¿Qué es un Conditional Inference Tree?

Un **Conditional Inference Tree (CIT)** —en R, `partykit::ctree`— es una variante de CART introducida por **Hothorn, Hornik & Zeileis (2006)**. La idea central:

> En cada nodo, antes de buscar el mejor split, hacer un **test estadístico de independencia** entre cada predictor $X_j$ y la variable respuesta $Y$. Si ningún predictor rechaza la hipótesis nula de independencia (después de corregir por comparaciones múltiples), el nodo **no se parte**.

Eso resuelve dos problemas clásicos de CART que ya vimos en el Tutorial 11:

| Problema de CART | Solución de CIT |
|---|---|
| Sesgo hacia variables con **muchos niveles** (más cortes posibles → más fácil que aparezcan en algún split) | Se elige primero la variable con mejor *p-value* del test de permutación; el número de niveles entra en la corrección del test, no en la elección |
| *Pruning* post-hoc con `ccp_alpha` y CV (caro) | *Pruning* implícito: si el test no rechaza, no se parte. El hiperparámetro es el nivel de significancia $\alpha$ |
| Sin teoría de inferencia estándar para los splits | Cada split tiene asociado un *p-value* (de permutación) interpretable |

**Algoritmo en una oración:** en cada nodo (1) elegir la variable $X_{j^*}$ con menor *p-value* contra $Y$; (2) si $p > \alpha$, parar; si no, encontrar el mejor split sobre $X_{j^*}$ y recursar.

El sesgo de la fila 1 es muy concreto. Vamos a verlo en acción en la Actividad 1.

## 2. ¿Existe algo así en Python?

**Respuesta corta:** sí, el paquete [`citrees`](https://github.com/rmill040/citrees) de R. Mills. Es un proyecto Python que implementa la misma familia de algoritmos: árboles cuya selección de variable y de punto de corte se basa en **tests estadísticos de permutación**.

**Caveats:**

- `citrees` **no es un port directo** de `partykit::ctree` — el propio README lo aclara. Implementa los mismos principios (permutation tests + stopping basado en significancia) con defaults y selectores propios.
- Para uso serio en un paper que cite específicamente `ctree`, lo estándar sigue siendo R + `partykit`. Una alternativa muy usada es llamar a R desde Python con `rpy2`.
- Para los visualizadores avanzados de árbol (intervalos, distribución por hoja), `partykit` sigue siendo superior.

### Instalación (opcional — para reproducir el notebook)

> ⚠️ **Estos comandos NO se ejecutan dentro del notebook.** Son comandos de **terminal del sistema operativo**. Si los pegás en una celda de código de Jupyter y apretás Shift+Enter, vas a obtener un `SyntaxError`. **Para seguir la clase no necesitás reproducir esto** — los outputs ya están guardados.

**Requisitos:**
- Python ≥ 3.12 (el paquete declara esta versión mínima)
- R instalado en el sistema (por la dependencia `rpy2`)

#### Mac / Linux (Terminal)

Parado en la carpeta `Tutorial 13/`:

```bash
# 1. Crear venv aislado. Reemplazar python3 por la versión ≥ 3.12 que tengas
python3 -m venv .venv
source .venv/bin/activate

# 2. Clonar citrees adentro de la carpeta y instalar en modo editable
git clone https://github.com/rmill040/citrees.git citrees_src
pip install -e ./citrees_src

# 3. Resto de dependencias para correr este notebook
pip install ISLP jupyter matplotlib seaborn

# 4. Registrar un kernel de Jupyter para usar desde el notebook
python -m ipykernel install --user --name tut13_ctree --display-name "Python (Tut13 CTree)"
```

#### Windows (PowerShell)

Parado en la carpeta `Tutorial 13/`:

```powershell
# 1. Crear venv aislado
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1

# 2. Clonar citrees y instalar en modo editable
git clone https://github.com/rmill040/citrees.git citrees_src
pip install -e .\citrees_src

# 3. Resto de dependencias
pip install ISLP jupyter matplotlib seaborn

# 4. Registrar kernel
python -m ipykernel install --user --name tut13_ctree --display-name "Python (Tut13 CTree)"
```

> En Windows, además de Python y R, hace falta tener Git instalado y, para `rpy2`, definir la variable de entorno `R_HOME` apuntando a la instalación de R. Si nunca configuraste `rpy2` en Windows, este paso suele ser el más fastidioso.

Después, al abrir el notebook, elegir el kernel **"Python (Tut13 CTree)"** desde el selector arriba a la derecha.

---

> 💡 **Si tenés problemas para reproducir esto, escribime y lo vemos juntos.** No te trabes acá — el notebook está pensado principalmente para ser leído con los outputs ya guardados.


## Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from ISLP import load_data

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

from citrees import ConditionalInferenceTreeRegressor
import citrees
print('citrees:', citrees.__version__)

RNG = 1

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


citrees: 0.1.0


## 3. Actividad 1 — El sesgo de selección de CART

**La idea de la actividad:** vamos a generar datos donde **$Y$ es puro ruido**, completamente independiente de todos los predictores. En ese escenario, *el árbol no debería partir nada* — no hay señal que capturar. Pero veremos que **CART igual parte y elige sistemáticamente las variables que ofrecen más puntos de corte posibles**, aunque $Y$ no dependa de ellas. Eso es el famoso sesgo de selección de variables. CIT, por construcción, evita ese problema.

### Diseño de la simulación

Generamos un dataset con $n = 500$ y cuatro predictores que se diferencian por **cuántos cortes posibles** ofrecen:

| Predictor | Tipo | Niveles distintos | Cortes posibles |
|---|---|---|---|
| `X_continua` | continua | ~500 (todos distintos) | hasta $n-1 \approx 499$ |
| `X_cat20` | categórica | 20 | hasta 19 |
| `X_cat5` | categórica | 5 | hasta 4 |
| `X_binaria` | binaria | 2 | 1 |

Y construimos $Y$ como **ruido normal**, sin relación con ninguna variable: $Y \sim \mathcal{N}(0, 1)$.

**Hipótesis teórica:** como CART busca *el mejor* corte entre todos los posibles, las variables con **más cortes posibles** tienen más chances de "ganar" por azar. El orden esperado del sesgo es:
$$\text{X\_continua} \;>\; \text{X\_cat20} \;>\; \text{X\_cat5} \;>\; \text{X\_binaria}.$$
CIT, en cambio, debería *no* splitear casi nunca (el test de permutación no rechaza la independencia).

In [3]:
def generar_datos_nulos(n=500, seed=0):
    """Genera X con 4 predictores de distinta cardinalidad e Y independiente."""
    rng = np.random.default_rng(seed)
    X = np.column_stack([
        rng.normal(0, 1, n),              # X_continua
        rng.binomial(1, 0.5, n),          # X_binaria
        rng.integers(0, 5, n),            # X_cat5
        rng.integers(0, 20, n),           # X_cat20
    ])
    y = rng.normal(0, 1, n)               # Y es puro ruido, NO depende de X
    return X, y

nombres_X = ['X_continua', 'X_binaria', 'X_cat5', 'X_cat20']

# Una sola corrida para inspección
X_demo, y_demo = generar_datos_nulos(n=500, seed=0)
print('Shape X:', X_demo.shape, ' | shape y:', y_demo.shape)
print('Correlación entre Y y cada predictor:')
for j, nombre in enumerate(nombres_X):
    print(f'  {nombre:12s}: corr = {np.corrcoef(X_demo[:, j], y_demo)[0, 1]:+.3f}')

Shape X: (500, 4)  | shape y: (500,)
Correlación entre Y y cada predictor:
  X_continua  : corr = -0.033
  X_binaria   : corr = +0.016
  X_cat5      : corr = +0.011
  X_cat20     : corr = -0.039


Las correlaciones son chicas y de signo aleatorio: $Y$ efectivamente no depende de ninguna variable. Ahora veamos qué hace cada árbol.

In [4]:
# CART (sin restricciones de profundidad)
cart = DecisionTreeRegressor(random_state=RNG).fit(X_demo, y_demo)
print('CART:')
print(f'  - profundidad del árbol resultante : {cart.get_depth()}')
print(f'  - cantidad de hojas                 : {cart.get_n_leaves()}')
print(f'  - variable del split RAÍZ           : {nombres_X[cart.tree_.feature[0]]}')
print(f'  - importancias relativas            :')
for nombre, imp in zip(nombres_X, cart.feature_importances_):
    print(f'      {nombre:12s}: {imp:.3f}')

CART:
  - profundidad del árbol resultante : 23
  - cantidad de hojas                 : 500
  - variable del split RAÍZ           : X_continua
  - importancias relativas            :
      X_continua  : 0.562
      X_binaria   : 0.059
      X_cat5      : 0.123
      X_cat20     : 0.257


In [5]:
# CIT (alpha = 0.05)
cit = ConditionalInferenceTreeRegressor(
    selector='pc', splitter='mse',
    alpha_selector=0.05, alpha_splitter=0.05,
    min_samples_split=20,
    random_state=RNG, verbose=0,
).fit(X_demo, y_demo)

# Profundidad y cantidad de hojas no son atributos directos de citrees,
# pero podemos chequear cuántas predicciones únicas devuelve sobre el train:
preds_unicas_cit = len(np.unique(cit.predict(X_demo)))
preds_unicas_cart = len(np.unique(cart.predict(X_demo)))

print('CIT (alpha=0.05):')
print(f'  - cantidad de regiones (predicciones únicas) : {preds_unicas_cit}')
print()
print('CART:')
print(f'  - cantidad de regiones (predicciones únicas) : {preds_unicas_cart}')
print()
print('Cuanto más cerca de 1 estén las regiones únicas, MENOS está partiendo el árbol')
print('(1 región = no partió nada = correctamente reconoció que Y es ruido).')

CIT (alpha=0.05):
  - cantidad de regiones (predicciones únicas) : 1

CART:
  - cantidad de regiones (predicciones únicas) : 500

Cuanto más cerca de 1 estén las regiones únicas, MENOS está partiendo el árbol
(1 región = no partió nada = correctamente reconoció que Y es ruido).


**Lectura:** CART parte el espacio en docenas de regiones aunque no haya señal. CIT lo deja en 1 sola región (o muy pocas), porque el test de permutación no rechaza la independencia.

### Simulación: ¿en qué variable parte CART el nodo raíz?

Repetimos el experimento muchas veces y vemos qué variable elige CART para el primer split. Si CART no tuviera sesgo, debería elegir cualquiera de las cuatro variables con probabilidad $\approx 1/4 = 25\%$. Si tiene sesgo hacia la de mayor cardinalidad, vamos a verlo.

In [6]:
n_replicas = 200
split_raiz_cart = []
for s in range(n_replicas):
    Xs, ys = generar_datos_nulos(n=500, seed=s)
    cart_s = DecisionTreeRegressor(random_state=RNG).fit(Xs, ys)
    split_raiz_cart.append(nombres_X[cart_s.tree_.feature[0]])

freq_cart = pd.Series(split_raiz_cart).value_counts(normalize=True).reindex(nombres_X, fill_value=0)
tabla = pd.DataFrame({
    'Variable': nombres_X,
    'Niveles': ['continua', '2', '5', '20'],
    'Frecuencia split raíz CART (%)': (freq_cart.values * 100).round(1),
    'Esperado si no hay sesgo (%)': [25.0] * 4,
})
print(f'Simulación con {n_replicas} réplicas. Y independiente de X.\n')
print(tabla.to_string(index=False))

Simulación con 200 réplicas. Y independiente de X.

  Variable  Niveles  Frecuencia split raíz CART (%)  Esperado si no hay sesgo (%)
X_continua continua                            66.0                          25.0
 X_binaria        2                             2.5                          25.0
    X_cat5        5                             9.0                          25.0
   X_cat20       20                            22.5                          25.0


**Lectura.** El resultado confirma exactamente la hipótesis teórica del sesgo de selección:

1. **`X_continua` se lleva el ~66%** de las elecciones, muy por encima del 25% esperado sin sesgo. Es la que ofrece más cortes posibles (uno entre cada par consecutivo de valores únicos).
2. **`X_cat20`** queda segunda con ~22%.
3. **`X_cat5`** y **`X_binaria`** quedan muy por debajo.

El patrón es monótono en la cantidad de cortes posibles. Y todo esto pasa aunque $Y$ es **estrictamente independiente** de todas las variables. CART está "encontrando" señal donde no la hay, y la encuentra preferentemente en los predictores que le dan más libertad para buscar.

### ¿Y CIT? ¿Cuántas veces parte el árbol cuando no hay señal?

Si CIT funciona bien, **el test de permutación no debería rechazar la independencia**, así que la raíz no se parte. Bajo $H_0$ verdadera, esperamos que CIT splittee como mucho un $\alpha = 5\%$ de las veces.

In [7]:
# La simulación con CIT es más lenta. Hacemos 50 réplicas para mostrar el patrón.
n_replicas_cit = 50
cit_splitio = []
for s in range(n_replicas_cit):
    Xs, ys = generar_datos_nulos(n=500, seed=s)
    cit_s = ConditionalInferenceTreeRegressor(
        selector='pc', splitter='mse',
        alpha_selector=0.05, alpha_splitter=0.05,
        min_samples_split=20, random_state=RNG, verbose=0,
    ).fit(Xs, ys)
    # Si la raíz no se parte, todas las predicciones son iguales (la media de y)
    cit_splitio.append(len(np.unique(cit_s.predict(Xs))) > 1)

tasa_split_cit = np.mean(cit_splitio) * 100
tasa_split_cart = 100.0  # CART siempre splittea cuando puede

print(f'En {n_replicas_cit} réplicas con Y = ruido puro:')
print(f'  CART splittea:  {tasa_split_cart:.1f}% de las veces')
print(f'  CIT splittea:   {tasa_split_cit:.1f}% de las veces')
print(f'  (Esperado bajo H0 verdadera: ≈ alpha = 5%)')

En 50 réplicas con Y = ruido puro:
  CART splittea:  100.0% de las veces
  CIT splittea:   2.0% de las veces
  (Esperado bajo H0 verdadera: ≈ alpha = 5%)


**Mensaje de la Actividad 1:**

- CART **siempre** parte el árbol (en la corrida única vimos profundidad 23 con 500 hojas para 500 observaciones — sobreajuste extremo), y prefiere las variables con más puntos de corte posibles aunque no haya señal real.
- CIT, gracias al test de permutación, **no parte cuando no debe**. Su tasa de falsos positivos está controlada por $\alpha$.

Esto importa en aplicaciones donde las variables tienen **cardinalidades muy distintas** — típicamente datos económicos con dummies + códigos administrativos + variables continuas mezcladas.

## 4. Actividad 2 — CART vs CIT sobre Boston Housing

Pasamos ahora a un dataset real, con $Y$ que **sí** depende de $X$. Acá las dos herramientas deberían dar resultados parecidos: la ventaja de CIT (no sesgarse) importa menos cuando las variables son casi todas continuas.

In [8]:
boston = load_data('Boston')
y = boston['medv'].values
X = boston.drop(columns=['medv'])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
print('Predictores (p):', X.shape[1])
print('Train:', X_train.shape, ' Test:', X_test.shape)

Predictores (p): 12
Train: (404, 12)  Test: (102, 12)


In [9]:
# CART con max_depth elegido por CV (igual que en Tut11/Tut12)
p = X_train.shape[1]
cart_cv = GridSearchCV(
    DecisionTreeRegressor(random_state=RNG),
    {'max_depth': list(range(1, p + 1))},
    cv=10, scoring='neg_mean_squared_error'
).fit(X_train, y_train)
best_cart = cart_cv.best_estimator_
mse_cart = mean_squared_error(y_test, best_cart.predict(X_test))
print(f'CART óptimo: max_depth = {best_cart.max_depth}, MSE test = {mse_cart:.2f}')

CART óptimo: max_depth = 6, MSE test = 30.28


In [10]:
# CIT con alpha = 0.05
cit_boston = ConditionalInferenceTreeRegressor(
    selector='pc', splitter='mse',
    alpha_selector=0.05, alpha_splitter=0.05,
    max_depth=p, min_samples_split=20,
    random_state=RNG, verbose=0,
).fit(X_train.values, y_train)
mse_cit = mean_squared_error(y_test, cit_boston.predict(X_test.values))
print(f'CIT (alpha=0.05): MSE test = {mse_cit:.2f}')

CIT (alpha=0.05): MSE test = 30.22


In [11]:
# Barrido de alpha: cuanto más chico, más conservador el árbol
filas = []
for alpha in [0.001, 0.01, 0.05, 0.1, 0.2]:
    cit_a = ConditionalInferenceTreeRegressor(
        selector='pc', splitter='mse',
        alpha_selector=alpha, alpha_splitter=alpha,
        max_depth=p, min_samples_split=20,
        random_state=RNG, verbose=0,
    ).fit(X_train.values, y_train)
    mse_a = mean_squared_error(y_test, cit_a.predict(X_test.values))
    filas.append({'alpha': alpha, 'MSE test': round(mse_a, 2)})
pd.DataFrame(filas)

,alpha,MSE test
0,0.001,33.23
1,0.010,31.07
2,0.050,30.22
3,0.100,30.71
4,0.200,29.38


In [12]:
# Comparación final
comparacion = pd.DataFrame({
    'Modelo': ['CART (sklearn)', 'CIT (citrees, alpha=0.05)'],
    'MSE test': [round(mse_cart, 2), round(mse_cit, 2)],
    'Cómo se controla el tamaño': ['max_depth por CV', 'alpha del test de permutación'],
    'Sesgo a variables multinivel': ['Sí (Actividad 1)', 'No (corregido)'],
})
print(comparacion.to_string(index=False))

                   Modelo  MSE test    Cómo se controla el tamaño Sesgo a variables multinivel
           CART (sklearn)     30.28              max_depth por CV             Sí (Actividad 1)
CIT (citrees, alpha=0.05)     30.22 alpha del test de permutación               No (corregido)


**Lectura del resultado en Boston.** Los MSE de CART y CIT son muy parecidos. Eso es coherente: los predictores de Boston son casi todos continuos, así que el sesgo de selección de CART (que vimos en la Actividad 1) acá no tiene mucho lugar para ejercerse. Donde CIT realmente brilla es en datasets con muchas dummies y categóricas, típicos de microdatos económicos (encuestas, registros administrativos).

## 5. Aplicación en economía: Huo & Feng (2024, IZA DP 16764)

**Cita:** Huo, Y. & Feng, S. (2024). *Childhood Circumstances and Health of American and Chinese Older Adults: A Machine Learning Evaluation of Inequality of Opportunity in Health*. IZA Discussion Paper No. 16764.

**Pregunta de investigación:** ¿cuánto de la desigualdad observada en salud entre adultos mayores se explica por **circunstancias de la infancia** (cosas que no son responsabilidad del individuo) versus **decisiones propias** (esfuerzo, estilo de vida)? Esta es la pregunta clásica de la literatura de *inequality of opportunity* (Roemer).

**Datos:**
- **HRS** (Health and Retirement Study) — Estados Unidos.
- **CHARLS** (China Health and Retirement Longitudinal Study) — China.
- Outcomes de salud: salud auto-reportada, limitaciones funcionales (ADL/IADL), enfermedades crónicas.
- Predictores de circunstancia: lugar de nacimiento rural/urbano, educación de los padres, salud de los padres, exposición a hambruna/guerra en la infancia, número de hermanos.

**Problema metodológico:** la literatura tradicional usa OLS sobre dummies de circunstancias. Eso impone una forma funcional aditiva y no captura interacciones. Si las circunstancias **interactúan** (p.ej., el efecto de tener padres poco educados depende del lugar de nacimiento), OLS lo pasa por alto.

**Cómo usan ML:** los autores emplean métodos basados en árboles (incluyendo *conditional inference forests*) para estimar la función $E[\text{salud} \mid \text{circunstancias}]$ sin imponer estructura. La varianza explicada de esa función es su medida de *inequality of opportunity*.

**Por qué CIT y no CART clásico:** porque las variables de circunstancia son mezcla de continuas y categóricas con varios niveles (educación de los padres, región), **justo el caso donde CART está sesgado, como vimos en la Actividad 1**. CIT no sufre ese sesgo, así que el ranking de importancia de variables es más confiable.

**Mensaje:** este paper ilustra un uso "sustantivo" de *conditional inference trees* en economía aplicada — no como predictor sino como **herramienta de descomposición de varianza** para una pregunta normativa: ¿cuánto de la desigualdad en salud es "injusta" en el sentido de Roemer?

## Referencias

- Huo, Y., & Feng, S. (2024). Childhood Circumstances and Health of American and Chinese Older Adults: A Machine Learning Evaluation of Inequality of Opportunity in Health. *IZA Discussion Paper No. 16764*. https://www.iza.org/publications/dp/16764
- Mills, R. (s.f.). `citrees`: Conditional Inference Trees in Python. https://github.com/rmill040/citrees